# Project 1B: Real Word2Vec Training (WikiText)
Train **real Skip-Gram Word2Vec from scratch** on WikiText-2 or WikiText-103. CPU Colab; no pretrained embeddings. WikiText-103 takes much longer and more RAM.

In [ ]:
!pip -q install 'gensim>=4.3.3' 'datasets>=3.0' scikit-learn matplotlib

## 1. Settings
Switch to `wikitext-103-raw-v1` for a larger training corpus.

In [ ]:
DATASET_SIZE='wikitext-2-raw-v1' # or 'wikitext-103-raw-v1'
DIM=100
WINDOW=5
MIN_COUNT=3 if '2-' in DATASET_SIZE else 5
EPOCHS=10 if '2-' in DATASET_SIZE else 5
SEED=42

## 2. Download and tokenize the Wikipedia-derived training text

In [ ]:
from datasets import load_dataset
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np,time
raw=load_dataset('Salesforce/wikitext',DATASET_SIZE,split='train')
def tokenize(line):
    if not line.strip() or line.lstrip().startswith('='):return []
    return simple_preprocess(line,deacc=True,min_len=2,max_len=25)
sentences=[t for s in raw['text'] if len(t:=tokenize(s))>=3]
print('Sentences:',len(sentences),'Tokens:',sum(map(len,sentences)))
print('Example:',sentences[0][:25])

## 3. Train Skip-Gram with Negative Sampling
`sg=1` means Skip-Gram, `negative=10` means ten sampled negatives. Track the approximate training loss after each epoch.

In [ ]:
model=Word2Vec(vector_size=DIM,window=WINDOW,min_count=MIN_COUNT,sg=1,negative=10,hs=0,workers=2,seed=SEED,compute_loss=True)
model.build_vocab(sentences)
print('Vocabulary:',len(model.wv))
losses=[]
start=time.time()
for epoch in range(EPOCHS):
    alpha0=0.025+(0.001-0.025)*epoch/EPOCHS
    alpha1=0.025+(0.001-0.025)*(epoch+1)/EPOCHS
    model.train(sentences,total_examples=model.corpus_count,epochs=1,start_alpha=alpha0,end_alpha=alpha1,compute_loss=True)
    # get_latest_training_loss() is the loss reported for the latest train() call.
    loss=model.get_latest_training_loss()
    losses.append(loss)
    print(f'Epoch {epoch+1}/{EPOCHS}: approximate loss={loss:,.0f}; elapsed={time.time()-start:.0f}s')
plt.plot(range(1,EPOCHS+1),losses,marker='o');plt.xlabel('Epoch');plt.ylabel('Approximate loss');plt.grid(alpha=.3);plt.show()

## 4. Nearest neighbors and cosine similarity
Small corpora can produce noisy relationships. Inspect results rather than assuming they will be meaningful.

In [ ]:
for word in ['king','queen','computer','science','city','music']:
    if word in model.wv:print(word,model.wv.most_similar(word,topn=6))
for a,b in [('king','queen'),('cat','dog'),('science','computer')]:
    if a in model.wv and b in model.wv:print(a,b,round(float(model.wv.similarity(a,b)),3))

## 5. Vector arithmetic
Analogies like king − man + woman ≈ queen are **not guaranteed**, especially with WikiText-2.

In [ ]:
def analogy(positive,negative):
    missing=[w for w in positive+negative if w not in model.wv]
    if missing:print('Missing words:',missing)
    else:print('Analogy:',model.wv.most_similar(positive=positive,negative=negative,topn=5))
analogy(['king','woman'],['man'])
analogy(['paris','germany'],['france'])

## 6. Visualize 100D vectors in 2D with PCA
PCA preserves variance, not exact semantic distances.

In [ ]:
words=[w for w in ['king','queen','man','woman','dog','cat','animal','computer','software','science','music','song','city','country','paris','france'] if w in model.wv]
if len(words)>=3:
    xy=PCA(n_components=2).fit_transform(np.array([model.wv[w] for w in words]))
    plt.figure(figsize=(10,7));plt.scatter(xy[:,0],xy[:,1])
    for word,(x,y) in zip(words,xy):plt.annotate(word,(x,y),xytext=(5,5),textcoords='offset points')
    plt.title('Learned Word2Vec embeddings — PCA');plt.grid(alpha=.2);plt.show()

## 7. Save the trained model and vectors

In [ ]:
model.save('wikitext_word2vec.model')
model.wv.save('wikitext_word_vectors.kv')
print('Saved model and vectors. Download via Colab Files sidebar.')

## Student challenges
1. Compare WikiText-2 and WikiText-103.
2. Change vector dimension 100 → 50 or 200.
3. Change context window 5 → 2 or 10.
4. Compare neighbors before and after more training epochs.
5. Why does training loss not guarantee a successful analogy?

**Notes:** Training time varies by Colab hardware. Results vary across runs with multiworker training.